# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and preprocess the FAIR² dataset using the `mlcroissant` library. All dataset entities such as record sets and fields are referenced by their unique `@id`s according to the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Install the mlcroissant library
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("Version:", metadata.version)
print("Published:", metadata.date_published)
print("Citation:", metadata.cite_as)

## 2. Data Overview
Review the available record sets, fields, and their `@id`s. The main FAIR² dataset exposes its record sets via Croissant metadata. Below, we enumerate the record set `@id`s and the associated field `@id`s.

In [ ]:
print("Available record sets in the dataset:")
record_sets = [rs["@id"] for rs in metadata.record_sets]
for rs in metadata.record_sets:
    print(f"- Record Set @id: {rs['@id']} | Name: {rs.get('name', '')}")
    print("  Fields (by @id):")
    for fld in rs['fields']:
        print(f"    - Field @id: {fld['@id']} | Name: {fld.get('name','')}")
    print()
# For demonstration, pick the first record set for example operations
example_record_set_id = record_sets[0] if record_sets else None

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Load the data from each record set into a DataFrame
dataframes = {}
loaded = False
for rs in metadata.record_sets:
    record_set_id = rs['@id']
    print(f"Loading records for Record Set '@id': {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  -> {len(df)} records loaded.")
        if not loaded:
            # Use the first record set for demonstration and EDA
            main_record_set_id = record_set_id
            loaded = True
    except Exception as e:
        print(f"  !! Failed to load: {e}")
print("")

if loaded:
    print("Columns in main record set (by field `@id`):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record set data could be loaded.")

## 4. Exploratory Data Analysis (EDA)
Carry out initial analysis and processing. For EDA, let's:
* Select a numeric field by its field `@id` (e.g., typical mass spec data would have protein abundance, molecular weight, peptide counts, etc.)
* Filter by threshold
* Normalize a numeric field
* Optionally group by a categorical field (if present)

In [ ]:
import numpy as np
# === Identify a numeric field and a group-by field by @id ===
if loaded:
    numeric_candidates = []
    for col in dataframes[main_record_set_id].columns:
        # Heuristically select numeric fields by checking dtype of the column
        if pd.api.types.is_numeric_dtype(dataframes[main_record_set_id][col]):
            numeric_candidates.append(col)
    print("Numeric candidate fields (by @id):", numeric_candidates)
    numeric_field_id = numeric_candidates[0] if numeric_candidates else None

    groupby_candidates = []
    for col in dataframes[main_record_set_id].columns:
        if pd.api.types.is_object_dtype(dataframes[main_record_set_id][col]) and col != numeric_field_id:
            groupby_candidates.append(col)
    groupby_field_id = groupby_candidates[0] if groupby_candidates else None

    if numeric_field_id is not None:
        # Filter for values > threshold (e.g., top expressions)
        threshold = dataframes[main_record_set_id][numeric_field_id].mean() if dataframes[main_record_set_id][numeric_field_id].notnull().any() else 0
        filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} rows")
        
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} (first few rows):")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by groupby_field_id if available
        if groupby_field_id is not None and groupby_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(groupby_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {groupby_field_id}:")
            display(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No loaded data for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields. Here, we plot the distribution of the chosen numeric field and, if available, grouped means for categories.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if loaded and numeric_field_id is not None:
    plt.figure(figsize=(8,5))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if groupby_field_id is not None and groupby_field_id in dataframes[main_record_set_id]:
        plt.figure(figsize=(10,5))
        order = dataframes[main_record_set_id][groupby_field_id].value_counts().index[:10]
        sns.boxplot(x=groupby_field_id, y=numeric_field_id, data=dataframes[main_record_set_id], order=order)
        plt.title(f"{numeric_field_id} distribution by {groupby_field_id} (top 10)")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
- Loaded the FAIR² protein dataset and listed its record sets and fields by their Croissant `@id`s.
- Extracted data for analysis, showing fields, example records, and performed basic filtering and normalization on a numeric field.
- Visualized numeric data distribution and groupwise summaries.

Further analysis can leverage additional fields (referenced by `@id`) and combine more advanced machine learning methods.